# 12 - Hyperparameter Tuning

In this notebook, we move beyond tuning a single parameter and explore automated strategies for optimizing an entire model: Grid Search and Random Search.

## Concept Overview
* **Grid Search**: Exhaustively tests every possible combination in a predefined grid. Very slow.
* **Random Search**: Randomly samples combinations. Statistically proven to find better models faster than Grid Search.

## Scikit-Learn Implementation: Grid Search
Let's tune a Random Forest.

In [ ]:
import numpy as np
import time
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

X, y = load_breast_cancer(return_X_y=True)
model = RandomForestClassifier(random_state=42)

# Define the Grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

# 3 x 3 x 3 = 27 combinations. 5 Folds = 135 models trained.
print("Starting Grid Search...")
start = time.time()
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X, y)
print(f"Grid Search Time: {time.time() - start:.2f} seconds")

print("Best Parameters:", grid_search.best_params_)
print(f"Best CV Accuracy: {grid_search.best_score_:.4f}")

## Scikit-Learn Implementation: Random Search
Instead of specific lists, we give Random Search probability distributions (like `randint`).

In [ ]:
from scipy.stats import randint

# Define the Distribution
param_dist = {
    'n_estimators': randint(50, 250),
    'max_depth': [None, 10, 20, 30, 40],
    'min_samples_split': randint(2, 15)
}

# We tell it to try exactly 20 random combinations
print("Starting Random Search...")
start = time.time()
random_search = RandomizedSearchCV(model, param_distributions=param_dist, n_iter=20, cv=5, scoring='accuracy', n_jobs=-1, random_state=42)
random_search.fit(X, y)
print(f"Random Search Time: {time.time() - start:.2f} seconds")

print("Best Parameters:", random_search.best_params_)
print(f"Best CV Accuracy: {random_search.best_score_:.4f}")

## Visualizing the Search Results
We can extract the `cv_results_` dictionary from the search object to see how every single combination performed.

In [ ]:
import pandas as pd

# Convert grid search results to a DataFrame
results = pd.DataFrame(grid_search.cv_results_)

# Keep only the important columns and sort by rank
results = results[['rank_test_score', 'mean_test_score', 'param_max_depth', 'param_n_estimators', 'param_min_samples_split']]
results = results.sort_values('rank_test_score')

display(results.head(5))

## Industry Notes
In modern ML, we rarely use Scikit-Learn for tuning deep models like XGBoost. Instead, we use **Bayesian Optimization** libraries like `Optuna` or `Hyperopt`. These libraries use mathematical models to 'learn' from previous attempts, focusing the search on the most promising areas of the hyperparameter space.

## Exercises
1. Modify the `param_grid` to include `criterion: ['gini', 'entropy']`. Run the grid search again. How many total models are trained now?
2. Research `Optuna`. Try writing a basic Optuna study to tune the `RandomForestClassifier` on this dataset.